In [1]:
import nibabel as nib
import numpy as np
from skimage import transform
import SimpleITK as sitk

def load_nifti(filepath):
    """Load a NIfTI MRI scan and return as a numpy array."""
    img = nib.load(filepath)          # Load .nii or .nii.gz file
    data = img.get_fdata()            # Shape: (W, H, D) for 3D volume
    return data, img.affine           # affine = spatial coordinates mapping

def normalize_mri(volume):
    """Z-score normalization: mean=0, std=1. Standard for MRI.
    This removes intensity bias between different scanners.
    """
    mean = np.mean(volume)
    std  = np.std(volume)
    return (volume - mean) / (std + 1e-8)

def resize_volume(volume, target_shape=(128, 128, 128)):
    """Resize all volumes to the same shape for CNN input."""
    # zoom factors along each axis
    factors = [t/s for t, s in zip(target_shape, volume.shape)]
    return transform.resize(volume, target_shape, mode='constant')

def skull_strip(volume_path):
    """Remove non-brain tissue using SimpleITK's brain extractor."""
    img = sitk.ReadImage(volume_path)
    stripped = sitk.BrainExtraction(img)     # Remove skull, scalp
    return sitk.GetArrayFromImage(stripped)

def full_pipeline(nifti_path):
    volume, affine = load_nifti(nifti_path)
    volume = skull_strip(nifti_path)         # Remove skull
    volume = normalize_mri(volume)           # Normalize intensity
    volume = resize_volume(volume)           # Resize to 128³
    return volume.astype(np.float32)

In [1]:
# model.py — 3D U-Net for Brain Tumor Segmentation
# Uses MONAI library — medical imaging PyTorch extension

from monai.networks.nets import UNet
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
import torch
from torch.optim import Adam

# ── Model Definition ──────────────────────────────────────────
model = UNet(
    spatial_dims=3,              # 3D (not 2D slices — full volume!)
    in_channels=4,               # BraTS uses 4 MRI modalities
    out_channels=3,              # 3 tumor subregions to segment
    channels=(16, 32, 64, 128, 256),  # Feature map sizes
    strides=(2, 2, 2, 2),         # Downsampling strides
    num_res_units=2               # Residual units per block
)

# ── Loss + Optimizer ──────────────────────────────────────────
loss_fn  = DiceLoss(to_onehot_y=True, softmax=True)
optimizer = Adam(model.parameters(), lr=1e-4)
metric    = DiceMetric(include_background=False, reduction="mean")

# ── Training Loop (simplified) ────────────────────────────────
def train_epoch(model, loader, device):
    model.train()
    total_loss = 0
    for batch in loader:
        images = batch["image"].to(device)   # (B, 4, 128, 128, 128)
        labels = batch["label"].to(device)   # (B, 1, 128, 128, 128)
        
        optimizer.zero_grad()
        outputs = model(images)               # Forward pass
        loss = loss_fn(outputs, labels)       # Dice loss
        loss.backward()                       # Backprop
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

# Run on Google Colab GPU for ~50 epochs (~4-6 hours on free T4)
# Expected Dice score: 0.75-0.85 on BraTS validation se

import numpy as np
from skimage.measure import marching_cubes
import trimesh
import torch

def predict_tumor_mask(model, volume, device):
    """Run inference → get 3D binary mask."""
    model.eval()
    with torch.no_grad():
        volume_tensor = torch.from_numpy(volume).unsqueeze(0).unsqueeze(0).to(device)
        output = model(volume_tensor)             # (1, 3, 128, 128, 128)
        mask = torch.argmax(output, dim=1).squeeze().cpu().numpy()
    return (mask > 0).astype(np.uint8)          # Binary: 1 = tumor

def mask_to_mesh(mask, output_path="tumor.stl", voxel_size=1.0):
    """Marching Cubes: 3D mask → triangle mesh."""
    # Find surface at threshold 0.5
    verts, faces, normals, _ = marching_cubes(mask, level=0.5, spacing=(voxel_size,)*3)
    
    # Create mesh with trimesh
    mesh = trimesh.Trimesh(vertices=verts, faces=faces, vertex_normals=normals)
    
    # Smooth the mesh (removes staircase artifacts from voxels)
    trimesh.smoothing.filter_laplacian(mesh, lamb=0.5, iterations=10)
    
    # Export as STL (Unity imports this directly)
    mesh.export(output_path)
    print(f"Mesh saved: {output_path} | {len(verts)} vertices, {len(faces)} faces")
    return output_path

In [9]:
import numpy as np
from skimage.measure import marching_cubes
import trimesh
import torch

def predict_tumor_mask(model, volume, device):
    """Run inference → get 3D binary mask."""
    model.eval()
    with torch.no_grad():
        volume_tensor = torch.from_numpy(volume).unsqueeze(0).unsqueeze(0).to(device)
        output = model(volume_tensor)             # (1, 3, 128, 128, 128)
        mask = torch.argmax(output, dim=1).squeeze().cpu().numpy()
    return (mask > 0).astype(np.uint8)          # Binary: 1 = tumor

def mask_to_mesh(mask, output_path="tumor.stl", voxel_size=1.0):
    """Marching Cubes: 3D mask → triangle mesh."""
    # Find surface at threshold 0.5
    verts, faces, normals, _ = marching_cubes(mask, level=0.5, spacing=(voxel_size,)*3)
    
    # Create mesh with trimesh
    mesh = trimesh.Trimesh(vertices=verts, faces=faces, vertex_normals=normals)
    
    # Smooth the mesh (removes staircase artifacts from voxels)
    trimesh.smoothing.filter_laplacian(mesh, lamb=0.5, iterations=10)
    
    # Export as STL (Unity imports this directly)
    mesh.export(output_path)
    print(f"Mesh saved: {output_path} | {len(verts)} vertices, {len(faces)} faces")
    return output_path